In [1]:
import cv2
import numpy as np
import time
from pymycobot.mecharm import MechArm

# --- 로봇 팔 초기화 ---
# 포트 이름은 환경에 맞게 변경 (예: "COM9")
mc = MechArm("COM9", 115200)

def init_robot():
    """로봇 팔을 안전한 초기 상태(Home 위치)로 이동"""
    print("[시스템] 로봇 팔을 초기 상태로 이동합니다.")
    mc.power_on()
    time.sleep(1)
    # 초기 각도 설정 (충돌을 방지하는 기본 Home 포지션)
    mc.send_angles([0, 0, 0, 0, 0, 0], 30)
    time.sleep(3)
    # 흡착 펌프 끄기 (Off = 1)
    mc.set_basic_output(5, 1)

# 초기화 실행
init_robot()

# --- 색상별 최종 이동 각도 설정 ---
TARGET_ANGLES = {
    "Red": [19.86, 57.65, -45.26, 0.17, 69.52, 50.53],
    "Green": [8.61, 45.08, -21.97, -1.05, 57.04, 39.99],
    "Blue": [-9.49, 43.5, -19.24, 4.92, 54.31, 39.63]
}

# --- 색상 감지 함수 ---
def color_detection(frame, color_name, lower_bound, upper_bound, min_box_size=500):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    
    if isinstance(lower_bound, list) and isinstance(upper_bound, list):
        mask1 = cv2.inRange(hsv, lower_bound[0], upper_bound[0])
        mask2 = cv2.inRange(hsv, lower_bound[1], upper_bound[1])
        mask = cv2.bitwise_or(mask1, mask2)
    else:
        mask = cv2.inRange(hsv, lower_bound, upper_bound)
    
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_DILATE, kernel)
    
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    detected_objects = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > min_box_size:
            x, y, w, h = cv2.boundingRect(cnt)
            center_x = x + w // 2
            center_y = y + h // 2
            detected_objects.append({
                "color": color_name,
                "box": (x, y, w, h),
                "center": (center_x, center_y),
                "area": area
            })
            
    return detected_objects

# --- HSV 범위 설정 ---
lower_red = [np.array([0, 100, 100]), np.array([170, 100, 100])]
upper_red = [np.array([10, 255, 255]), np.array([180, 255, 255])]
lower_green = np.array([35, 80, 80])
upper_green = np.array([85, 255, 255])
lower_blue = np.array([95, 100, 100])
upper_blue = np.array([130, 255, 255])

min_box_size = 5500

def pixel_to_robot_coords(px, py):
    """카메라 픽셀 좌표를 로봇 팔의 좌표(X, Y, Z)로 변환"""
    robot_x = 200 + (py - 240) * -0.5
    robot_y = 0 + (px - 320) * -0.5  
    robot_z = 50                      
    return robot_x, robot_y, robot_z

def pick_and_place(target_x, target_y, target_z, color_name):
    """대상 위치에서 물건을 잡고, 색상별 지정된 각도로 이동하여 놓기"""
    print(f"[{color_name}] 목표 좌표로 이동 중: X={target_x:.2f}, Y={target_y:.2f}")
    
    # 1. 대상 위치 위로 이동 후 하강
    mc.send_coords([target_x, target_y, target_z + 80, 0, 180, 0], 30, 0)
    time.sleep(2)
    mc.send_coords([target_x, target_y, target_z, 0, 180, 0], 20, 0)
    time.sleep(1.5)
    
    # 2. Suction Pump ON (0 = ON)
    mc.set_basic_output(5, 0)
    print("Suction Pump ON: 물체를 흡착합니다.")
    time.sleep(1.5) 
    
    # 3. 물체를 든 채로 위로 상승
    mc.send_coords([target_x, target_y, target_z + 80, 0, 180, 0], 30, 0)
    time.sleep(2)
    
    # 4. 지정된 색상별 최종 각도로 이동
    target_angles = TARGET_ANGLES.get(color_name, [0, 0, 0, 0, 0, 0])
    print(f"[{color_name}] 지정된 각도로 이동합니다: {target_angles}")
    mc.send_angles(target_angles, 30)
    time.sleep(3.5) # 각도 이동 대기
    
    # 5. 3초 대기 후 Suction Pump OFF (1 = OFF)
    print("3초 대기 후 물체를 놓습니다.")
    time.sleep(3)
    mc.set_basic_output(5, 1)
    time.sleep(1)
    
    # 6. 하나의 Task 완료 후 안정성을 위해 즉시 초기 위치로 복귀
    print(f"[{color_name}] 작업 완료. 다음 단계를 위해 초기 상태로 복귀합니다.")
    init_robot()
    time.sleep(1)


Note: This class is no longer maintained since v3.6.0, please refer to the project documentation: https://github.com/elephantrobotics/pymycobot/blob/main/README.md
[시스템] 로봇 팔을 초기 상태로 이동합니다.


In [2]:

# --- 메인 루프 ---
cap = cv2.VideoCapture(0)

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        red_objs = color_detection(frame, "Red", lower_red, upper_red, min_box_size)
        green_objs = color_detection(frame, "Green", lower_green, upper_green, min_box_size)
        blue_objs = color_detection(frame, "Blue", lower_blue, upper_blue, min_box_size)
        
        all_objects = red_objs + green_objs + blue_objs
        
        for obj in all_objects:
            x, y, w, h = obj["box"]
            color_name = obj["color"]
            box_colors = {"Red": (0, 0, 255), "Green": (0, 255, 0), "Blue": (255, 0, 0)}
            bgr = box_colors.get(color_name, (0, 255, 255))
            
            cv2.rectangle(frame, (x, y), (x + w, y + h), bgr, 2)
            cv2.putText(frame, color_name, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, bgr, 2)
            
        cv2.imshow('Color Detection & Pick-Place', frame)
        
        key = cv2.waitKey(1) & 0xFF
        
        # 'p' 키를 누르면 작업 시작
        if key == ord('p') and len(all_objects) > 0:
            print(f"\n--- 총 {len(all_objects)}개의 객체가 검출되었습니다. 순차 작업을 시작합니다. ---")
            
            for idx, obj in enumerate(all_objects):
                print(f"\n[{idx+1}/{len(all_objects)}] {obj['color']} 객체 Task 시작")
                px, py = obj["center"]
                
                # 픽셀을 로봇 좌표로 변환
                rx, ry, rz = pixel_to_robot_coords(px, py)
                
                # 색상 이름(color_name)을 함께 넘겨주어 각도 제어
                pick_and_place(rx, ry, rz, obj['color'])
            
            print("\n모든 다중 객체 처리가 완료되었습니다. 대기 모드로 전환합니다.")
            
        elif key == ord('q'):
            break

finally:
    print("프로그램을 종료하며 초기 상태로 복귀합니다.")
    init_robot()
    mc.release_all_servos()
    cap.release()
    cv2.destroyAllWindows()


--- 총 3개의 객체가 검출되었습니다. 순차 작업을 시작합니다. ---

[1/3] Red 객체 Task 시작
[Red] 목표 좌표로 이동 중: X=140.50, Y=-12.50
Suction Pump ON: 물체를 흡착합니다.
[Red] 지정된 각도로 이동합니다: [19.86, 57.65, -45.26, 0.17, 69.52, 50.53]
3초 대기 후 물체를 놓습니다.
[Red] 작업 완료. 다음 단계를 위해 초기 상태로 복귀합니다.
[시스템] 로봇 팔을 초기 상태로 이동합니다.

[2/3] Green 객체 Task 시작
[Green] 목표 좌표로 이동 중: X=198.00, Y=15.50
Suction Pump ON: 물체를 흡착합니다.
[Green] 지정된 각도로 이동합니다: [8.61, 45.08, -21.97, -1.05, 57.04, 39.99]
3초 대기 후 물체를 놓습니다.
[Green] 작업 완료. 다음 단계를 위해 초기 상태로 복귀합니다.
[시스템] 로봇 팔을 초기 상태로 이동합니다.

[3/3] Blue 객체 Task 시작
[Blue] 목표 좌표로 이동 중: X=244.50, Y=37.50
Suction Pump ON: 물체를 흡착합니다.
[Blue] 지정된 각도로 이동합니다: [-9.49, 43.5, -19.24, 4.92, 54.31, 39.63]
3초 대기 후 물체를 놓습니다.
[Blue] 작업 완료. 다음 단계를 위해 초기 상태로 복귀합니다.
[시스템] 로봇 팔을 초기 상태로 이동합니다.

모든 다중 객체 처리가 완료되었습니다. 대기 모드로 전환합니다.

--- 총 3개의 객체가 검출되었습니다. 순차 작업을 시작합니다. ---

[1/3] Red 객체 Task 시작
[Red] 목표 좌표로 이동 중: X=125.50, Y=8.00
Suction Pump ON: 물체를 흡착합니다.
[Red] 지정된 각도로 이동합니다: [19.86, 57.65, -45.26, 0.17, 69.52, 50.53]
3초 대기 후 물체를 놓습니다.
[Re

In [ ]:
import cv2
import numpy as np
import time
from pymycobot.mecharm import MechArm

# --- 로봇 팔 초기화 ---
# 포트 이름은 환경에 맞게 변경 (예: "COM9")
mc = MechArm("COM9", 115200)

def init_robot():
    """로봇 팔을 안전한 초기 상태(Home 위치)로 이동"""
    print("[시스템] 로봇 팔을 초기 상태로 이동합니다.")
    mc.power_on()
    time.sleep(1)
    # 초기 각도 설정 (충돌을 방지하는 기본 Home 포지션)
    mc.send_angles([0, 0, 0, 0, 0, 0], 30)
    time.sleep(3)
    # 흡착 펌프 끄기 (Off = 1)
    mc.set_basic_output(5, 1)

# 초기화 실행
init_robot()

# --- 물체를 놓을 목적지 각도 배열 (색상 무관, 순서대로 적용) ---
TARGET_ANGLES = [
    [19.86, 57.65, -45.26, 0.17, 69.52, 50.53], # 위치 1
    [8.61, 45.08, -21.97, -1.05, 57.04, 39.99], # 위치 2
    [-9.49, 43.5, -19.24, 4.92, 54.31, 39.63]   # 위치 3
]

# --- 색상 감지 함수 ---
def color_detection(frame, color_name, lower_bound, upper_bound, min_box_size=5500):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    
    if isinstance(lower_bound, list) and isinstance(upper_bound, list):
        mask1 = cv2.inRange(hsv, lower_bound[0], upper_bound[0])
        mask2 = cv2.inRange(hsv, lower_bound[1], upper_bound[1])
        mask = cv2.bitwise_or(mask1, mask2)
    else:
        mask = cv2.inRange(hsv, lower_bound, upper_bound)
    
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_DILATE, kernel)
    
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    detected_objects = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > min_box_size:
            x, y, w, h = cv2.boundingRect(cnt)
            center_x = x + w // 2
            center_y = y + h // 2
            detected_objects.append({
                "color": color_name,
                "box": (x, y, w, h),
                "center": (center_x, center_y),
                "area": area
            })
            
    return detected_objects

# --- HSV 범위 설정 ---
lower_red = [np.array([0, 100, 100]), np.array([170, 100, 100])]
upper_red = [np.array([10, 255, 255]), np.array([180, 255, 255])]
lower_green = np.array([35, 80, 80])
upper_green = np.array([85, 255, 255])
lower_blue = np.array([95, 100, 100])
upper_blue = np.array([130, 255, 255])

# 최소 인식 면적 변경 (5500)
min_box_size = 5500

def pixel_to_robot_coords(px, py):
    """카메라 픽셀 좌표를 로봇 팔의 좌표(X, Y, Z)로 변환"""
    robot_x = 200 + (py - 240) * -0.5
    robot_y = 0 + (px - 320) * -0.5  
    robot_z = 50                      
    return robot_x, robot_y, robot_z

def pick_and_place(target_x, target_y, target_z, place_angles, color_name, task_num):
    """대상 위치에서 물건을 잡고, 지정된 순서의 각도로 이동하여 놓기"""
    print(f"[{color_name}] 대상 인지 완료. 픽(Pick)을 위해 X={target_x:.2f}, Y={target_y:.2f}로 이동합니다.")
    
    # 1. 대상 위치 위로 이동 후 하강
    mc.send_coords([target_x, target_y, target_z + 80, 0, 180, 0], 30, 0)
    time.sleep(2)
    mc.send_coords([target_x, target_y, target_z, 0, 180, 0], 20, 0)
    time.sleep(1.5)
    
    # 2. Suction Pump ON (0 = ON)
    mc.set_basic_output(5, 0)
    print("Suction Pump ON: 물체를 흡착합니다.")
    time.sleep(1.5) 
    
    # 3. 물체를 든 채로 위로 상승
    mc.send_coords([target_x, target_y, target_z + 80, 0, 180, 0], 30, 0)
    time.sleep(2)
    
    # 4. 순서에 맞는 플레이스(Place) 각도로 이동
    print(f"[Task {task_num}] 지정된 위치 각도로 이동합니다: {place_angles}")
    mc.send_angles(place_angles, 30)
    time.sleep(3.5) # 각도 이동 대기
    
    # 5. 3초 대기 후 Suction Pump OFF (1 = OFF)
    print("3초 대기 후 물체를 놓습니다.")
    time.sleep(3)
    mc.set_basic_output(5, 1)
    time.sleep(1)
    
    # 6. 하나의 Task 완료 후 안정성을 위해 즉시 초기 위치로 복귀
    print(f"[Task {task_num} 완료] 다음 단계를 위해 초기 상태로 복귀합니다.")
    init_robot()
    time.sleep(1)


# --- 메인 루프 ---
cap = cv2.VideoCapture(0)

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        red_objs = color_detection(frame, "Red", lower_red, upper_red, min_box_size)
        green_objs = color_detection(frame, "Green", lower_green, upper_green, min_box_size)
        blue_objs = color_detection(frame, "Blue", lower_blue, upper_blue, min_box_size)
        
        # 검출된 모든 객체 리스트 통합
        all_objects = red_objs + green_objs + blue_objs
        
        for obj in all_objects:
            x, y, w, h = obj["box"]
            color_name = obj["color"]
            box_colors = {"Red": (0, 0, 255), "Green": (0, 255, 0), "Blue": (255, 0, 0)}
            bgr = box_colors.get(color_name, (0, 255, 255))
            
            cv2.rectangle(frame, (x, y), (x + w, y + h), bgr, 2)
            cv2.putText(frame, color_name, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, bgr, 2)
            
        cv2.imshow('Color Detection & Pick-Place', frame)
        
        key = cv2.waitKey(1) & 0xFF
        
        # 'p' 키를 누르면 작업 시작
        if key == ord('p') and len(all_objects) > 0:
            print(f"\n--- 총 {len(all_objects)}개의 대상(볼)이 인지되었습니다. 순차적 Pick & Place를 시작합니다. ---")
            
            # 검출된 객체 수만큼 반복하면서 지정된 순서 각도로 이동
            for idx, obj in enumerate(all_objects):
                task_num = idx + 1
                color_name = obj['color']
                
                # 순서에 맞는 각도 가져오기 (배열 인덱스 초과 시 순환하도록 처리)
                place_angles = TARGET_ANGLES[idx % len(TARGET_ANGLES)]
                
                print(f"\n[{task_num}/{len(all_objects)}] {color_name} 대상 작업 준비 중...")
                px, py = obj["center"]
                
                # 픽셀을 로봇 좌표로 변환
                rx, ry, rz = pixel_to_robot_coords(px, py)
                
                # 순차적 목적지 각도(place_angles)를 넘겨주어 실행
                pick_and_place(rx, ry, rz, place_angles, color_name, task_num)
            
            print("\n모든 작업이 완료되었습니다. 대기 모드로 전환합니다.")
            
        elif key == ord('q'):
            break

finally:
    print("프로그램을 종료하며 초기 상태로 복귀합니다.")
    init_robot()
    mc.release_all_servos()
    cap.release()
    cv2.destroyAllWindows()